<img src="../data/zls.svg" width="120" style="float:right; margin: 0 20px 10px 20px;">

# Answer Sheet

Example solutions and explanations for the challenges in each notebook.

These are **not** the only way to solve each challenge — if your approach works, it's valid!

---

## 01 — TTS Basics

### Easy — Change the text

Just swap the `text` variable and re-run from Step 2. Everything else stays the same.

In [ ]:
text = "Ech hunn d'Sproochmaschinn immens gär."
model = "claude"

### Medium — Inspect the result metadata

The `result` dictionary contains more than just the audio. Print it to see what metadata the API sends back.

In [ ]:
# After polling completes, inspect the full result:
print(result.keys())

# Look inside the "result" key for metadata:
for key, value in result["result"].items():
    if key != "data":  # skip the audio data itself
        print(f"{key}: {value}")

Try this with a short sentence and then a longer one — you should see the `duration` increase.

### Stretch — Two texts, two files

We just repeat the submit → poll → decode → save flow twice with different text and filenames.

In [ ]:
import base64
import time
import requests
from IPython.display import Audio

BASE_URL = "https://sproochmaschinn.lu"

session = requests.post(f"{BASE_URL}/api/session").json()
session_id = session["session_id"]

texts = {
    "greeting.wav": "Moien, wéi geet et?",
    "weather.wav": "Wéi ass d'Wieder haut?",
}

for filename, text in texts.items():
    # Submit
    tts_response = requests.post(
        f"{BASE_URL}/api/tts/{session_id}",
        json={"text": text, "model": "claude"}
    ).json()
    request_id = tts_response["request_id"]

    # Poll
    for _ in range(60):
        result = requests.get(f"{BASE_URL}/api/result/{request_id}").json()
        if result["status"] == "completed":
            break
        time.sleep(1)
    else:
        raise TimeoutError(f"TTS for '{filename}' did not complete in time.")

    # Decode and save
    audio_bytes = base64.b64decode(result["result"]["data"])
    with open(filename, "wb") as f:
        f.write(audio_bytes)
    print(f"Saved: {filename}")

In [ ]:
display(Audio("greeting.wav"))
display(Audio("weather.wav"))

---

## 02 — STT Basics

### Easy — Disable speaker identification

Change the `enable_speaker_identification` parameter to `"false"` in the STT submit call.

```python
data={"enable_speaker_identification": "false"}
```

When you inspect the results, the speaker-related fields (like `speaker` in the words DataFrame) will either be missing or set to a default value, since the model no longer tries to figure out who said what.

### Medium — Filter low-confidence words

Use a pandas boolean filter to find words where the model was less sure.

In [ ]:
# Assuming words_df is already created from notebook 02:
low_confidence = words_df[words_df["score"] < 0.8]
print(f"{len(low_confidence)} words with confidence below 0.8:")
low_confidence

You might notice that short words, names, or dialect-specific words tend to have lower confidence — the model is less sure about things it doesn't see as often in training data.

### Stretch — TTS then STT round-trip

Generate a WAV with TTS, then feed it back into STT and see if the transcription matches the original text.

In [ ]:
import base64
import time
import requests

BASE_URL = "https://sproochmaschinn.lu"

session = requests.post(f"{BASE_URL}/api/session").json()
session_id = session["session_id"]

# Step 1: Generate audio from text
original_text = "Moien, wéi geet et?"

tts_response = requests.post(
    f"{BASE_URL}/api/tts/{session_id}",
    json={"text": original_text, "model": "claude"}
).json()

for _ in range(60):
    result = requests.get(f"{BASE_URL}/api/result/{tts_response['request_id']}").json()
    if result["status"] == "completed":
        break
    time.sleep(1)
else:
    raise TimeoutError("TTS request did not complete in time.")

audio_bytes = base64.b64decode(result["result"]["data"])
with open("roundtrip.wav", "wb") as f:
    f.write(audio_bytes)
print("TTS done, saved roundtrip.wav")

# Step 2: Transcribe that audio
with open("roundtrip.wav", "rb") as f:
    stt_response = requests.post(
        f"{BASE_URL}/api/stt/{session_id}",
        files={"audio": f}
    ).json()

for _ in range(120):
    result = requests.get(f"{BASE_URL}/api/result/{stt_response['request_id']}").json()
    if result["status"] == "completed":
        break
    time.sleep(1)
else:
    raise TimeoutError("STT request did not complete in time.")

transcribed_text = result["result"]["text"]

# Step 3: Compare
print(f"Original:    {original_text}")
print(f"Transcribed: {transcribed_text}")
print(f"Match: {original_text.lower().strip() == transcribed_text.lower().strip()}")

Don't worry if the match isn't exact — small differences in punctuation or spelling are normal. The interesting thing is seeing how close the round-trip gets.

---

## 03 — Exports and Analysis

### Easy — Change the export level

Change `"sentence"` to `"word"` in the export params:

```python
params={
    "level": "word",
    "include_speakers": "true",
    "include_confidence": "true"
}
```

Instead of getting one entry per sentence, you'll get one entry per word — each with its own timestamp, speaker, and confidence score.

### Medium — Words per speaker

Use `groupby` to count how many words each speaker said.

In [ ]:
# Assuming words_df is already created from the notebook:
if "speaker" in words_df.columns:
    words_per_speaker = words_df.groupby("speaker").size()
    print(words_per_speaker)
    print(f"\nMost talkative: {words_per_speaker.idxmax()} with {words_per_speaker.max()} words")
else:
    print("No speaker column — make sure speaker identification is enabled.")

### Stretch — Speaking time per speaker

Each word has a `start` and `end` timestamp. We can compute the total time each speaker spent talking by summing the duration of each word.

In [ ]:
if "speaker" in words_df.columns and "start" in words_df.columns and "end" in words_df.columns:
    words_df["word_duration"] = words_df["end"] - words_df["start"]
    time_per_speaker = words_df.groupby("speaker")["word_duration"].sum()
    print("Total speaking time per speaker (seconds):")
    print(time_per_speaker)
else:
    print("Missing required columns — make sure speaker identification is enabled.")

Note: this sums the time of individual words, so pauses between words aren't counted. It's an approximation of how much each person spoke.

---

## 04 — Advanced: Helper Functions

### 1. `summarise_result`

A simple function that prints the key information from a completed result.

In [ ]:
def summarise_result(result):
    # Print a short summary of a completed API result.
    print(f"Status: {result['status']}")

    data = result.get("result", {})

    if "duration" in data:
        print(f"Duration: {data['duration']}s")
    if "total_words" in data:
        print(f"Words: {data['total_words']}")
    if "total_segments" in data:
        print(f"Segments: {data['total_segments']}")
    if "text" in data:
        preview = data["text"][:80] + "..." if len(data["text"]) > 80 else data["text"]
        print(f"Text: {preview}")

In [ ]:
# Example usage (assumes you have a completed result from the notebook):
# summarise_result(result)

### 2. `words_to_dataframe`

Takes the transcription dict and returns the words as a DataFrame.

In [ ]:
import pandas as pd


def words_to_dataframe(transcription):
    # Convert the 'words' list from a transcription result into a DataFrame.
    return pd.DataFrame(transcription["words"])

In [ ]:
# Example usage:
# df = words_to_dataframe(result["result"])
# df.head()

---

## 05 — Advanced: Build a Small Client

### What happens if you call `client.tts(...)` before `create_session()`?

`self.session_id` is `None`, so the request URL becomes something like `.../api/tts/None`. The API doesn't recognise that as a valid session, so it returns an error. You'll likely see a `KeyError` when trying to access `data["request_id"]` from the error response, or a `requests` exception.

This shows why it's important to always create a session first — and in a production version, you might want the class to check for this and raise a clear error message.

---

## 06 — Advanced Challenges

### Challenge 1 — Compare TTS outputs

In [ ]:
import base64
import time
import requests

BASE_URL = "https://sproochmaschinn.lu"

session = requests.post(f"{BASE_URL}/api/session").json()
session_id = session["session_id"]

texts = [
    "Moien.",
    "Moien, wéi geet et dir?",
]

for text in texts:
    tts_response = requests.post(
        f"{BASE_URL}/api/tts/{session_id}",
        json={"text": text, "model": "claude"}
    ).json()

    for _ in range(60):
        result = requests.get(f"{BASE_URL}/api/result/{tts_response['request_id']}").json()
        if result["status"] == "completed":
            break
        time.sleep(1)
    else:
        raise TimeoutError("TTS request did not complete in time.")

    metadata = {k: v for k, v in result["result"].items() if k != "data"}
    print(f"Text: {text}")
    for k, v in metadata.items():
        print(f"  {k}: {v}")
    print()

### Challenge 2 — Speaker statistics

In [ ]:
import requests
import time
import pandas as pd
from pathlib import Path

BASE_URL = "https://sproochmaschinn.lu"
AUDIO_PATH = Path("../data/sample_audio.wav")

# Transcribe with speaker identification
session = requests.post(f"{BASE_URL}/api/session").json()
session_id = session["session_id"]

with open(AUDIO_PATH, "rb") as f:
    stt_response = requests.post(
        f"{BASE_URL}/api/stt/{session_id}",
        files={"audio": f},
        data={"enable_speaker_identification": "true"}
    ).json()

for _ in range(120):
    result = requests.get(f"{BASE_URL}/api/result/{stt_response['request_id']}").json()
    if result["status"] == "completed":
        break
    time.sleep(1)
else:
    raise TimeoutError("STT request did not complete in time.")

words_df = pd.DataFrame(result["result"]["words"])

# Stats
if "speaker" in words_df.columns:
    print("Average confidence per speaker:")
    print(words_df.groupby("speaker")["score"].mean())

    print("\nWord count per speaker:")
    print(words_df.groupby("speaker").size())

    words_df["word_duration"] = words_df["end"] - words_df["start"]
    print("\nTotal speaking time per speaker (seconds):")
    print(words_df.groupby("speaker")["word_duration"].sum())

### Challenge 3 — Batch transcription

In [ ]:
import requests
import time
from pathlib import Path

BASE_URL = "https://sproochmaschinn.lu"


def batch_transcribe(audio_paths):
    # Transcribe a list of audio files and return a list of results.
    session = requests.post(f"{BASE_URL}/api/session").json()
    session_id = session["session_id"]
    results = []

    for path in audio_paths:
        path = Path(path)
        print(f"Transcribing {path.name}...")

        with open(path, "rb") as f:
            stt_response = requests.post(
                f"{BASE_URL}/api/stt/{session_id}",
                files={"audio": f}
            ).json()

        for _ in range(120):
            result = requests.get(f"{BASE_URL}/api/result/{stt_response['request_id']}").json()
            if result["status"] == "completed":
                break
            time.sleep(1)
        else:
            raise TimeoutError(f"STT for {path.name} did not complete in time.")

        results.append(result)
        print(f"  Done: {result['result']['text'][:60]}...")

    return results


# Example usage:
# results = batch_transcribe(["../data/sample_audio.wav"])

### Challenge 4 — End-to-end pipeline

This is very similar to the Stretch challenge in notebook 02. Text goes into TTS, the audio comes out, then goes into STT, and we compare.

In [ ]:
import base64
import time
import requests

BASE_URL = "https://sproochmaschinn.lu"


def end_to_end(text, model="claude"):
    # Text -> TTS -> WAV -> STT -> compare.
    session = requests.post(f"{BASE_URL}/api/session").json()
    session_id = session["session_id"]

    # TTS
    tts_resp = requests.post(
        f"{BASE_URL}/api/tts/{session_id}",
        json={"text": text, "model": model}
    ).json()

    for _ in range(60):
        result = requests.get(f"{BASE_URL}/api/result/{tts_resp['request_id']}").json()
        if result["status"] == "completed":
            break
        time.sleep(1)
    else:
        raise TimeoutError("TTS request did not complete in time.")

    # Save WAV
    audio_bytes = base64.b64decode(result["result"]["data"])
    wav_path = "pipeline_output.wav"
    with open(wav_path, "wb") as f:
        f.write(audio_bytes)
    print(f"TTS saved to {wav_path}")

    # STT
    with open(wav_path, "rb") as f:
        stt_resp = requests.post(
            f"{BASE_URL}/api/stt/{session_id}",
            files={"audio": f}
        ).json()

    for _ in range(120):
        result = requests.get(f"{BASE_URL}/api/result/{stt_resp['request_id']}").json()
        if result["status"] == "completed":
            break
        time.sleep(1)
    else:
        raise TimeoutError("STT request did not complete in time.")

    transcribed = result["result"]["text"]

    print(f"Original:    {text}")
    print(f"Transcribed: {transcribed}")
    print(f"Match: {text.lower().strip() == transcribed.lower().strip()}")

    return result


# Example:
# end_to_end("Moien, wéi geet et?")

### Challenge 5 — Improve the client

Here's an extended version of the client from notebook 05 with retries, an `export` method, and a `tts_to_file` convenience method.

In [ ]:
import base64
import requests
import time
from pathlib import Path


class ImprovedClient:
    def __init__(self, base_url="https://sproochmaschinn.lu", max_retries=3):
        self.base_url = base_url
        self.max_retries = max_retries
        self.session_id = None

    def _request(self, method, url, **kwargs):
        # Make a request with automatic retries on network errors.
        for attempt in range(self.max_retries):
            try:
                response = requests.request(method, url, **kwargs)
                response.raise_for_status()
                return response.json()
            except requests.RequestException as e:
                if attempt == self.max_retries - 1:
                    raise
                print(f"Retry {attempt + 1}/{self.max_retries} after error: {e}")
                time.sleep(2)

    def create_session(self):
        data = self._request("POST", f"{self.base_url}/api/session")
        self.session_id = data["session_id"]
        return self.session_id

    def tts(self, text, model="claude"):
        data = self._request(
            "POST",
            f"{self.base_url}/api/tts/{self.session_id}",
            json={"text": text, "model": model}
        )
        return data["request_id"]

    def stt(self, audio_path, enable_speaker_identification=True):
        with open(audio_path, "rb") as f:
            data = self._request(
                "POST",
                f"{self.base_url}/api/stt/{self.session_id}",
                files={"audio": f},
                data={"enable_speaker_identification": str(enable_speaker_identification).lower()}
            )
        return data["request_id"]

    def poll(self, request_id, sleep_seconds=1, max_polls=120):
        for _ in range(max_polls):
            result = self._request("GET", f"{self.base_url}/api/result/{request_id}")
            if result["status"] == "completed":
                return result
            if result["status"] == "failed":
                raise RuntimeError(f"Request failed: {result}")
            time.sleep(sleep_seconds)
        raise TimeoutError(f"Request {request_id} did not complete in time.")

    def export(self, request_id, level="sentence", include_speakers=True, include_confidence=True):
        """Fetch a structured export of a completed transcription."""
        return self._request(
            "GET",
            f"{self.base_url}/api/result/{request_id}/export/timestamps",
            params={
                "level": level,
                "include_speakers": str(include_speakers).lower(),
                "include_confidence": str(include_confidence).lower(),
            }
        )

    def tts_to_file(self, text, output_path, model="claude"):
        # Generate speech and save it directly to a file.
        request_id = self.tts(text, model=model)
        result = self.poll(request_id)
        audio_bytes = base64.b64decode(result["result"]["data"])
        output_path = Path(output_path)
        output_path.write_bytes(audio_bytes)
        print(f"Saved to {output_path}")
        return output_path

In [ ]:
# Example usage:
# client = ImprovedClient()
# client.create_session()
# client.tts_to_file("Moien!", "improved_output.wav")